# ✦ Business-SK · AI stylist test (Qwen-Image-Edit-2511, free)

Turns a scraped product photo (e.g. a model wearing a hoodie) into a **styled product photo** — the garment laid flat on dark stone with a palm leaf, like the Instagram post you shared.

Model: **Qwen-Image-Edit-2511** + **Lightning 4-step** (both Apache-2.0 — commercial use OK), compressed to 4-bit while loading so it fits a free T4.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. The first run downloads ~60 GB (≈15 min). Step 4 prints the time per image and shows the results.

In [ ]:
# 1 · GPU + memory
import subprocess, psutil
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("RAM: %.1f GB" % (psutil.virtual_memory().total / 1e9))

In [ ]:
# 2 · Libraries (same diffusers as the laptop + LoRA support)
%pip install -q "diffusers==0.40.0" "transformers==5.17.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" peft

In [ ]:
# 3 · Load Qwen-Image-Edit-2511 in 4-bit + the Lightning 4-step LoRA (download ≈15 min the first time)
import time, torch
from diffusers import QwenImageEditPlusPipeline, PipelineQuantizationConfig
t = time.time()
q = PipelineQuantizationConfig(quant_backend="bitsandbytes_4bit",
                               quant_kwargs={"load_in_4bit": True, "bnb_4bit_quant_type": "nf4",
                                             "bnb_4bit_compute_dtype": torch.float16},
                               components_to_quantize=["transformer", "text_encoder"])
pipe = QwenImageEditPlusPipeline.from_pretrained("Qwen/Qwen-Image-Edit-2511", quantization_config=q,
                                                 torch_dtype=torch.float16, low_cpu_mem_usage=True)
pipe.load_lora_weights("lightx2v/Qwen-Image-Edit-2511-Lightning",
                       weight_name="Qwen-Image-Edit-2511-Lightning-4steps-V1.0-fp32.safetensors")
pipe.enable_model_cpu_offload()
print(f"✔ loaded in {time.time() - t:.0f}s")

In [ ]:
# 4 · Style two real scraped products and time it
import io, urllib.request
import numpy as np
from PIL import Image
from IPython.display import display

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return Image.open(io.BytesIO(urllib.request.urlopen(req, timeout=30).read())).convert("RGB")

PROMPT = ("Take only the garment the person is wearing and show it as a styled product photo: the garment laid flat, "
          "neatly arranged with natural fabric folds, on a dark textured stone floor, a green palm leaf in the top corner, "
          "soft natural light from the top left, realistic soft shadows, top-down view. No person, no body, no mannequin. "
          "Keep the exact colours, logo, print, zip, pockets and every detail of the garment unchanged.")

for url in ["https://m.media-amazon.com/images/I/61ZEES1vVjL.jpg", "https://m.media-amazon.com/images/I/514RyNBuKIL.jpg"]:
    src = fetch(url)
    t = time.time()
    out = pipe(image=[src], prompt=PROMPT, num_inference_steps=4, true_cfg_scale=1.0,
               height=1280, width=1024, generator=torch.Generator("cpu").manual_seed(7)).images[0]
    a = np.asarray(out).astype("float32")
    print(f"{url.rsplit('/', 1)[-1]}: {time.time() - t:.0f}s | black {(a.max(axis=2) < 8).mean():.0%}")
    sheet = Image.new("RGB", (1024 + 820, 1280), "black")
    sheet.paste(src.resize((800, int(800 * src.height / src.width))).crop((0, 0, 800, 1280)), (0, 0))
    sheet.paste(out, (820, 0))
    display(sheet.resize((922, 640)))